In [ ]:
import torchvision.transforms as transforms
from PIL import Image

# Define augmentation transforms
augmentation_transforms = transforms.Compose([
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    # Add more augmentation transforms as needed
])

# Apply augmentation and save images
for i in range(len(train_data0.dataset)):
    image, target = train_data0.dataset[i]
    augmented_image = augmentation_transforms(image)
    
    # Save augmented image to disk
    augmented_image.save(f"augmentExperiment/image_{i}.jpg")


In [1]:
import os
import torch
from torchvision import transforms
from PIL import Image

class YOLOFormatDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        
        self.image_files = sorted(os.listdir(image_dir))
        self.label_files = sorted(os.listdir(label_dir))
        assert len(self.image_files) == len(self.label_files), "Number of images and labels must match"
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.image_files[idx])
        label_path = os.path.join(self.label_dir, self.label_files[idx])
        
        # Read image
        image = Image.open(image_path).convert("RGB")
        
        # Read YOLO format label file and parse bounding boxes
        with open(label_path, 'r') as file:
            lines = file.readlines()
            boxes = [list(map(float, line.strip().split())) for line in lines]
        
        # Apply transformations
        if self.transform:
            image, boxes = self.transform(image, boxes)
        
        return image, boxes

def apply_augmentation(image, boxes):
    # Define transformations
    transform = transforms.Compose([
        transforms.ToTensor(), 
        transforms.RandomRotation(degrees=10),
        transforms.RandomHorizontalFlip(p=0.5),# Convert image to tensor
        # Add more transformations as needed
    ])
    
    # Apply transformations
    transformed_image = transform(image)
    
    return transformed_image, boxes


In [17]:
import os
import torch
from torchvision import transforms
from PIL import Image

class YOLOFormatDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, label_dir, transform=None, save_dir=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.save_dir = save_dir
        
        self.image_files = sorted(os.listdir(image_dir))
        self.label_files = sorted(os.listdir(label_dir))
        print(len(self.image_files),len(self.label_files))
        #assert len(self.image_files) == len(self.label_files)
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.image_files[idx])
        label_path = os.path.join(self.label_dir, self.label_files[idx])
        
        # Read image
        image = Image.open(image_path).convert("RGB")
        
        # Read YOLO format label file and parse bounding boxes
        with open(label_path, 'r') as file:
            lines = file.readlines()
            boxes = [list(map(float, line.strip().split())) for line in lines]
        
        # Apply transformations
        if self.transform:
            transformed_image, transformed_boxes = self.transform(image, boxes)
            
            # Save augmented image and labels
            if self.save_dir:
                self.save_augmented_data(transformed_image, transformed_boxes, idx)
                
            return transformed_image, transformed_boxes
        
        return image, boxes

    def save_augmented_data(self, image, boxes, idx):
        if self.save_dir:
            os.makedirs(self.save_dir, exist_ok=True)
            
            # Save augmented image
            image_save_path = os.path.join(self.save_dir, f"image_{idx}.jpg")
            image.save(image_save_path)
            
            # Save augmented labels
            label_save_path = os.path.join(self.save_dir, f"labels_{idx}.txt")
            with open(label_save_path, 'w') as file:
                for box in boxes:
                    file.write(" ".join(map(str, box)) + "\n")

def apply_augmentation(image, boxes):
    # Define transformations
    transform = transforms.Compose([
        transforms.RandomRotation(degrees=10.42),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Adjust color
        transforms.RandomResizedCrop(size=(640, 640), scale=(0.8, 1.0)),  # Random resized crop
        transforms.RandomGrayscale(p=0.1),
    ])
    
    # Apply transformations
    transformed_image = transform(image)
    
    return transformed_image, boxes

# Example usage:
image_dir = "./archive/resized640yolo/images/test"
label_dir = "./archive/resized640yolo/labels/test"
save_dir1= "./augmentExperiment/images"
save_dir2= "./augmentExperiment/labels"
save_dir = "./augmentExperiment"
# Create an instance of the dataset with augmentation and save directory
dataset = YOLOFormatDataset(image_dir=image_dir, label_dir=label_dir, transform=apply_augmentation, save_dir=save_dir)

# Iterate through the dataset to apply augmentation and save the augmented data
for idx, (image, boxes) in enumerate(dataset):
    # Augmented image and labels are saved to the specified save directory
    print(boxes)  # Add additional processing if needed


19 20


IsADirectoryError: [Errno 21] Is a directory: './archive/resized640yolo/labels/test/.ipynb_checkpoints'

In [ ]:
# Define the directory paths for images and labels
image_dir = "./archive/resized640yolo/images/test"
label_dir = "./archive/resized640yolo/labels/test"

# Create an instance of the dataset
dataset = YOLOFormatDataset(image_dir=image_dir, label_dir=label_dir)


In [ ]:
# Apply augmentation and save images
for i in range(len(train_data0.dataset)):
    image, target = train_data0.dataset[i]
    augmented_image = augmentation_transforms(image)
    
    # Save augmented image to disk
    augmented_image.save(f"augmentExperiment/image_{i}.jpg")

In [ ]:
import os
import json
import torch
from torchvision.transforms.functional import to_tensor
from PIL import Image

class YOLOFormatJSONDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir, images_dir, json_annotation_file, input_dim=(640, 640), transform=None):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.json_annotation_file = json_annotation_file
        self.input_dim = input_dim
        self.transform = transform
        
        # Load JSON annotations
        with open(os.path.join(data_dir, json_annotation_file), 'r') as f:
            self.annotations = json.load(f)
        
        # List of image paths
        self.image_files = [os.path.join(images_dir, image_info['file_name']) for image_info in self.annotations['images']]
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        # Read image
        image_path = os.path.join(self.data_dir, self.image_files[idx])
        image = Image.open(image_path).convert("RGB")
        
        # Get bounding box annotations for the current image
        image_id = self.annotations['images'][idx]['id']
        boxes = []
        for annotation in self.annotations['annotations']:
            if annotation['image_id'] == image_id:
                # Convert YOLO format to x_center, y_center, width, height format
                box = [
                    annotation['bbox'][0] + annotation['bbox'][2] / 2,  # x_center
                    annotation['bbox'][1] + annotation['bbox'][3] / 2,  # y_center
                    annotation['bbox'][2],  # width
                    annotation['bbox'][3]   # height
                ]
                boxes.append(box)
        
        # Apply transformations
        if self.transform:
            image, boxes = self.transform(image, boxes)
        
        return image, boxes

def apply_augmentation(image, boxes):
    # Define transformations
    transform = transforms.Compose([
        transforms.Resize((640, 640)),  # Resize images to input_dim
        transforms.ToTensor(),          # Convert image to tensor
        # Add more transformations as needed
    ])
    
    # Apply transformations
    transformed_image = transform(image)
    
    return transformed_image, boxes
